# **`pd.pivot()` Method:**

The `pd.pivot()` method is used to reshape data — it transforms or `"pivots"` a DataFrame by rearranging its rows and columns based on unique values in specific columns.

**Syntax:**

```python
DataFrame.pivot(index=None, columns=None, values=None)
```

| Parameter | Description                                |
| --------- | ------------------------------------------ |
| `index`   | Column to use to make new frame’s index.   |
| `columns` | Column to use to make new frame’s columns. |
| `values`  | Column(s) to populate the values.          |

**Returns:** A new DataFrame reshaped according to the index/column/value combinations.

**Important:**

* It works only **when your data is unique** across the combination of index and columns.
* If there are duplicates, it will raise a `ValueError`. For such cases, use `pivot_table()` instead.

It’s useful when:

* You want to **reshape "long-form" (tidy) data into "wide-form" data**.

* You need to create a table format (like pivot tables in Excel) for analysis or visualization.

* You want to summarize or view the data in a structured table by spreading values across columns.

### **Example 1: Basic Pivot:**

In [9]:
#### Original DataFrame (long-form):

import pandas as pd

df = pd.DataFrame({
    'date': ['2023-01', '2023-01', '2023-02', '2023-02'],
    'product': ['A', 'B', 'A', 'B'],
    'sales': [100, 150, 120, 180]
})

print(df)

      date product  sales
0  2023-01       A    100
1  2023-01       B    150
2  2023-02       A    120
3  2023-02       B    180


#### **Apply `pivot()`:**

In [10]:
pivot_df = df.pivot(index='date', columns='product', values='sales')
print(pivot_df)

product    A    B
date             
2023-01  100  150
2023-02  120  180


* `index='date'` → rows become dates
* `columns='product'` → columns become products (A, B)
* `values='sales'` → values are filled with sales numbers

### **Example 2: Using multiple values columns (not allowed with `pivot()`)**

If we have more than one value column and want to pivot both, `pivot()` won’t work. You’ll need `pivot_table()` instead.

In [11]:
df2 = pd.DataFrame({
    'department': ['HR', 'HR', 'IT', 'IT'],
    'month': ['Jan', 'Feb', 'Jan', 'Feb'],
    'revenue': [1000, 1100, 1500, 1600],
    'cost': [200, 220, 300, 310]
})
df2

,department,month,revenue,cost
0,HR,Jan,1000,200
1,HR,Feb,1100,220
2,IT,Jan,1500,300
3,IT,Feb,1600,310


In [12]:
# If we try this:

df2.pivot(index='month', columns='department', values=['revenue', 'cost'])

revenue       cost     
department      HR    IT   HR   IT
month                             
Feb           1100  1600  220  310
Jan           1000  1500  200  300

### ⚠️ **What if there are duplicate entries?**

In [16]:
df3 = pd.DataFrame({
    'month': ['Jan', 'Jan'],
    'product': ['A', 'A'],
    'sales': [100, 200]
})
print(df3)
print("+"*45)
print(df3.pivot(index='month', columns='product', values='sales'))

  month product  sales
0   Jan       A    100
1   Jan       A    200
+++++++++++++++++++++++++++++++++++++++++++++


ValueError: Index contains duplicate entries, cannot reshape

**Solution?** Use `pivot_table()` with aggregation.

In [17]:
df3.pivot_table(index='month', columns='product', values='sales', aggfunc='sum')

product,A
month,
Jan,300


| Feature                                   | `pivot()` |
| ----------------------------------------- | --------- |
| Works like Excel Pivot Table              | ✅         |
| Only works with unique index/column pairs | ✅         |
| Cannot aggregate automatically            | ❌         |
| Simple and fast for clean data            | ✅         |
| Replaces rows with column values          | ✅         |

### **Real-World Use Cases:**
1. **Sales Data**: Reshape monthly sales per product into columns to compare trends.

2. **Web Analytics**: Convert daily data into a format where each page or metric is a column.

3. **Health Data**: Pivot patient data so each test result type becomes a column.

4. **Finance Reports**: Convert transactions into a table where each type of expense becomes a column.

------
---
---
----

## **`pd.pivot()` & `pd.crosstab()` Methods:**

| Term                  | In pandas?          | Purpose                                     | Summary                                                                 |
| --------------------- | ------------------- | ------------------------------------------- | ----------------------------------------------------------------------- |
| **Crosstab**          | ✅ Yes               | Summarize counts or aggregate values        | Quick way to count combinations of categorical variables (like a table) |
| **Pivot Table**       | ✅ Yes               | Reshape and aggregate with more flexibility | Excel-style summary tables with more options                            |
| **Contingency Table** | ❌ No (not a method) | Statistical term for a table of frequencies | Can be created using `crosstab` (or `pivot_table` in special cases)     |

### 1. **`pd.crosstab()`** → **Contingency Table Generator:**

* Special function for generating **`contingency tables`** (counts of combinations).

* Typically used for **categorical variable analysis**.

* Good for **`confusion matrices`**, demographic counts, etc.

```python
pd.crosstab(df['Gender'], df['Smoker'])
```

Think of it as:

> 📊 "How many Females are Smokers vs Non-Smokers?"



### 2. **`pd.pivot_table()`** → **Flexible Table Generator:**

* More **`flexible and powerful`** than `crosstab`.
* Allows multiple **`aggregation functions`** (`mean`, `sum`, etc.).
* Supports **`multi-indexing`** and complex reshaping.

```python
pd.pivot_table(df, index='Gender', columns='Smoker', values='Age', aggfunc='mean')
```

Think of it as:

> 📈 "What’s the **average age** of Male/Female by Smoking status?"

---

### 3. **Contingency Table** → **Statistical Concept:**

* Not a pandas function, just a **concept**.
* A table showing **frequency distribution** of two variables.
* You **create** it using `crosstab` in pandas.

In statistics:

```text
              Predicted
              Yes   No
Actual  Yes   50    10
        No    5     35
```

This is a 2x2 contingency table.

## **Summary of Differences:**

| Feature           | Crosstab                           | Pivot Table                 | Contingency Table    |
| ----------------- | ---------------------------------- | --------------------------- | -------------------- |
| Built-in function | Yes (`pd.crosstab`)                | Yes (`pd.pivot_table`)      | No                   |
| Counts            | Yes (default)                      | No (need `aggfunc='count'`) | Yes                  |
| Aggregation       | Limited                            | Very flexible               | Counts only          |
| Values param      | Optional                           | Required if aggregating     | Not applicable       |
| Use case          | Frequency tables, confusion matrix | Summary stats by group      | Statistical analysis |
| Normalization     | Yes (`normalize=True`)             | No built-in                 | Depends on method    |

## **When to Use Each?**

| Task                                       | Use                                                 |
| ------------------------------------------ | --------------------------------------------------- |
| Count combinations of categories (fast)    | `pd.crosstab()`                                     |
| Build Excel-style summary table with stats | `pd.pivot_table()`                                  |
| Statistical test (like chi-squared test)   | Use `crosstab` to create contingency table for test |

## **Example:**

In [13]:
import pandas as pd

df = pd.DataFrame({
    'Gender': ['Male', 'Female', 'Female', 'Male', 'Female', 'Male'],
    'Smoker': ['Yes', 'No', 'No', 'Yes', 'Yes', 'No'],
    'Age': [25, 30, 22, 45, 28, 35]
})

In [14]:
### Using `crosstab`:

pd.crosstab(df['Gender'], df['Smoker'])

Smoker,No,Yes
Gender,,
Female,2,1
Male,1,2


In [15]:
### Using `pivot_table` for average Age:

pd.pivot_table(df, index='Gender', columns='Smoker', values='Age', aggfunc='mean')

Smoker,No,Yes
Gender,,
Female,26.0,28.0
Male,35.0,35.0


> 
> * **All three relate to summarizing data**, especially with categorical variables.
> 
> * **Crosstab** is best for **counts and categorical comparisons**.
> 
> * **Pivot tables** are for **advanced, multi-function aggregation**.
> 
> * **Contingency table** is the **theory/statistics** behind what `crosstab()` produces.
> 